# Stochastic Calculus, Step by Step · Lesson 21: The Greeks

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. An identity that does the work · 3. Delta · 4. Gamma · 5. Theta, and its balance with gamma · 6. The Greeks across prices · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 21,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. An identity that does the work

$S\varphi(d_+) = Ke^{-r\tau}\varphi(d_-)$, with $\tau = T - t$. It makes the messy terms cancel in every derivative below. Checked at several prices and times:

In [ ]:
id_rows = []
for s, tau in ((80.0, 1.0), (100.0, 1.0), (120.0, 1.0), (100.0, 0.25)):
    a_ = s * float(phi(d_plus(s, tau))); b_ = K * math.exp(-r * tau) * float(phi(d_minus(s, tau)))
    id_rows.append({"S": s, "tau": tau, "left": a_, "right": b_})
out["identity"] = {"rows": id_rows, "max_gap": max(abs(x["left"] - x["right"]) for x in id_rows)}
pd.DataFrame(id_rows)

## 3. Delta

$\Delta = \partial C/\partial S = N(d_+)$. Differentiating the formula gives $N(d_+)$ plus two terms that the identity cancels.

In [ ]:
tau0 = T
dp0, dm0 = float(d_plus(S0, tau0)), float(d_minus(S0, tau0))
delta_f = lambda s, tau: Nv(d_plus(s, tau))
gamma_f = lambda s, tau: phi(d_plus(s, tau)) / (s * sigma * np.sqrt(tau))
theta_f = lambda s, tau: -s * phi(d_plus(s, tau)) * sigma / (2 * np.sqrt(tau)) - r * K * np.exp(-r * tau) * Nv(d_minus(s, tau))
vega_f = lambda s, tau: s * phi(d_plus(s, tau)) * np.sqrt(tau)
h = 0.01
bump_delta = float((call(S0 + h, tau0) - call(S0 - h, tau0)) / (2 * h))
out["delta"] = {"formula": float(delta_f(S0, tau0)), "bump": bump_delta, "d_plus": dp0, "phi_plus": float(phi(dp0)),
                "phi_minus": float(phi(dm0)), "put": float(delta_f(S0, tau0)) - 1}
out["delta"]

## 4. Gamma

$\Gamma = \partial^2C/\partial S^2 = \varphi(d_+)/(S\sigma\sqrt\tau)$: how fast delta moves as the share moves, and so how often the hedge must be rebalanced. It is largest near the strike and grows as expiry nears.

In [ ]:
bump_gamma = float((call(S0 + h, tau0) - 2 * call(S0, tau0) + call(S0 - h, tau0)) / h**2)
out["gamma"] = {"formula": float(gamma_f(S0, tau0)), "bump": bump_gamma,
                "rows": [{"tau": t_, "at_strike": float(gamma_f(K, t_))} for t_ in (1.0, 0.25, 0.04, 0.01)],
                "delta_move_1": float(gamma_f(S0, tau0))}
pd.DataFrame(out["gamma"]["rows"])

## 5. Theta, and its balance with gamma

$\Theta = \partial C/\partial t = -\dfrac{S\varphi(d_+)\sigma}{2\sqrt\tau} - rKe^{-r\tau}N(d_-)$. The Black-Scholes equation (Lesson 19) ties it to gamma: $\Theta + rS\Delta + \tfrac12\sigma^2S^2\Gamma = rV$. A hedged option earns from big moves (gamma) and pays for the waiting (theta).

In [ ]:
k_ = 1e-5
bump_theta = float((call(S0, tau0 - k_) - call(S0, tau0 + k_)) / (2 * k_))
bal_rows = []
for s in (80.0, 100.0, 105.0, 120.0):
    v_ = float(call(s, tau0)); d_ = float(delta_f(s, tau0)); g_ = float(gamma_f(s, tau0)); th = float(theta_f(s, tau0))
    bal_rows.append({"S": s, "theta": th, "r_S_delta": r * s * d_, "half_gamma": 0.5 * sigma**2 * s**2 * g_, "rV": r * v_,
                     "gap": th + r * s * d_ + 0.5 * sigma**2 * s**2 * g_ - r * v_})
# one day at S = 100: what gamma earns against what theta costs
day = 1 / 252
out["theta"] = {"formula": float(theta_f(S0, tau0)), "bump": bump_theta, "per_day": float(theta_f(S0, tau0)) * day,
                "part_vol": float(-S0 * phi(dp0) * sigma / (2 * math.sqrt(tau0))), "part_rate": float(-r * K * math.exp(-r * tau0) * N(dm0)),
                "rows": bal_rows, "vega": float(vega_f(S0, tau0)), "vega_bump": float((call(S0, tau0, sigma + 1e-4) - call(S0, tau0, sigma - 1e-4)) / 2e-4),
                "break_even_move": sigma * S0 * math.sqrt(day), "break_even_pct": sigma * math.sqrt(day) * 100,
                "gamma_gain_at_be": 0.5 * float(gamma_f(S0, tau0)) * (sigma * S0 * math.sqrt(day)) ** 2}
pd.DataFrame(bal_rows)

## 6. The Greeks across prices

Delta runs from 0 to 1 and steepens near expiry; gamma is a hump around the strike, taller and narrower near expiry.

In [ ]:
Sg = np.linspace(60, 150, 181)
fig = {"S": Sg.tolist()}
for t_ in (1.0, 0.25, 0.04):
    key = f"t{int(round(t_ * 100)):03d}"
    fig[f"delta_{key}"] = delta_f(Sg, t_).tolist()
    fig[f"gamma_{key}"] = gamma_f(Sg, t_).tolist()
fig["ticks"] = [{"x": x, "label": str(x)} for x in (60, 80, 100, 120, 140)]
tab = [{"S": s, "call": float(call(s, tau0)), "delta": float(delta_f(s, tau0)), "gamma": float(gamma_f(s, tau0)),
        "theta": float(theta_f(s, tau0)), "vega": float(vega_f(s, tau0))} for s in (80.0, 90.0, 100.0, 105.0, 110.0, 120.0, 130.0)]
out["across"] = {"fig": fig, "rows": tab}
pd.DataFrame(tab)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Prove that $S\varphi(d_+) = Ke^{-r\tau}\varphi(d_-)$. (Write $d_+ = d_- + \sigma\sqrt\tau$, expand $d_+^2$, and use the definition of $d_-$.)

**Exercise 2.** Differentiate $C = SN(d_+) - Ke^{-r\tau}N(d_-)$ with respect to $S$, using the chain rule on each term, and use the identity to show $\Delta = N(d_+)$.

**Exercise 3.** Differentiate $\Delta$ to find $\Gamma = \varphi(d_+)/(S\sigma\sqrt\tau)$. Evaluate it at $S = 100$, $\tau = 1$. If the share rises by 1, roughly how much does delta change?

**Exercise 4.** Differentiate $C$ with respect to $t$ to find $\Theta$, and verify the Black-Scholes equation directly from the formulas for $\Theta$, $\Delta$ and $\Gamma$.

**Exercise 5.** Use put-call parity to find the put's delta, gamma and theta from the call's. Which of them are the same as the call's, and why?

**Exercise 6.** Show that **vega**, $\partial C/\partial\sigma$, equals $S\varphi(d_+)\sqrt\tau$, and that it equals $\sigma\tau S^2\Gamma$. Why are vega and gamma so closely related?

**Exercise 7.** (Code) Compute delta, gamma, theta and vega at $S = 80, 100, 120$ by small bumps of the formula and compare with the formulas. How does the error of the gamma bump change as the bump size runs through $1, 0.1, 0.01, 0.001, 0.0001$?

**Exercise 8.** (Code) Buy the call at $S = 100$, sell $\Delta$ shares, finance the difference at rate $r$, and let one trading day pass with moves of $0$, $\pm0.5\%$, $\pm1.26\%$ and $\pm2.5\%$. Compare the exact one-day profit with $\tfrac12\Gamma S^2\big((\delta S/S)^2 - \sigma^2\delta t\big)$. Where is the break-even move?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")